In [ ]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
naoconvertidos = pd.to_numeric(df['TotalCharges'], errors='coerce').isna().sum()
print(f"A coluna TotalCharges está com {naoconvertidos} linhas sem números.")

A coluna TotalCharges está com 11 linhas sem números.


In [ ]:
# Como são apenas 11 linhas de 7043, pode-se preencher os nulos com a mediana da coluna
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
mediana_total = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(mediana_total)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [ ]:
# Algumas colunas possuem varições de "não" como a coluna MultipleLines, que possui "No phone service" e "No".
# Já que para modelos de machine learning ambos significam a mesma coisa (o serviço não existe), é padronizado para apenas "No".

print("Antes:", df['MultipleLines'].unique())

colunas_servicos = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in colunas_servicos:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

print("Depois:", df['MultipleLines'].unique())

Antes: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
Depois: <StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [ ]:
# Transformar as colunas categóricas em binário (0 ou 1).

df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})
df['Partner'] = df['Partner'].map({'Yes': 1, 'No': 0})
df['Dependents'] = df['Dependents'].map({'Yes': 1, 'No': 0})
df['PhoneService'] = df['PhoneService'].map({'Yes': 1, 'No': 0})
df['PaperlessBilling'] = df['PaperlessBilling'].map({'Yes': 1, 'No': 0})
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# One-hot encoding
colunas_de_texto = df.select_dtypes(include=['str']).columns.tolist() # passa as colunas para um lista para ser manipulada

if 'customerID' in colunas_de_texto:
    colunas_de_texto.remove('customerID')

df = pd.get_dummies(df, columns=colunas_de_texto, drop_first=True, dtype=int) # drop_first=True para evitar multicolinearidade (uma coluna for 1, as outras serão 0).

In [ ]:
# A normalização/escala coloca todos na mesma proporção

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
colunas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,...,OnlineBackup_Yes,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,1,0,1,0,-1.277445,0,1,-1.160323,-0.994242,...,1,0,0,0,0,0,0,0,1,0
1,5575-GNVDE,0,0,0,0,0.066327,1,0,-0.259629,-0.173244,...,0,1,0,0,0,1,0,0,0,1
2,3668-QPYBK,0,0,0,0,-1.236724,1,1,-0.362660,-0.959674,...,1,0,0,0,0,0,0,0,0,1
3,7795-CFOCW,0,0,0,0,0.514251,0,0,-0.746535,-0.194766,...,0,1,1,0,0,1,0,0,0,0
4,9237-HQITU,1,0,0,0,-1.236724,1,1,0.197365,-0.940470,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
# Separação entre variaveis preditivas (X) e variavel alvo (y).

X = df.drop(columns=['customerID', 'Churn']) 
y = df['Churn']

In [ ]:
# Divisão entre treino e teste, com proporção de 80% para treino e 20% para teste.

from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Realização da criação e instaciamento do modelo de Random Forest

from sklearn.ensemble import RandomForestClassifier

modelo_churn = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
modelo_churn.fit(X_treino, y_treino)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

In [12]:
# Realiza a previsão do modelo com os dados de teste e exibe a acurácia geral e o relatório de classificação.
from sklearn.metrics import accuracy_score, classification_report

previsoes = modelo_churn.predict(X_teste)
print(f"Acurácia Geral: {accuracy_score(y_teste, previsoes):.2%}\n")
print(classification_report(y_teste, previsoes))

Acurácia Geral: 80.98%

              precision    recall  f1-score   support

           0       0.84      0.92      0.88      1036
           1       0.70      0.50      0.58       373

    accuracy                           0.81      1409
   macro avg       0.77      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409



In [ ]:
# Calcular a probabilidade de Churn para cada cliente do dataset

probabilidades = modelo_churn.predict_proba(X)[:, 1]

# Carregar o arquivo original de novo em uma variável limpa para manter os textos amigáveis

df_final = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Injetar a nova coluna na tabela.
df_final['Churn_Probability'] = probabilidades

# Arredondar para duas casas decimais
df_final['Churn_Probability'] = df_final['Churn_Probability'].round(2)

# Visualizar o resultado final
df_final[['customerID', 'Contract', 'MonthlyCharges', 'Churn', 'Churn_Probability']].head()

,customerID,Contract,MonthlyCharges,Churn,Churn_Probability
0,7590-VHVEG,Month-to-month,29.85,No,0.55
1,5575-GNVDE,One year,56.95,No,0.05
2,3668-QPYBK,Month-to-month,53.85,Yes,0.45
3,7795-CFOCW,One year,42.30,No,0.06
4,9237-HQITU,Month-to-month,70.70,Yes,0.67


In [ ]:
#Conexão com o banco de dados PostgreSQL
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env local
load_dotenv()

USUARIO = os.getenv("DB_USER")
SENHA = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORTA = os.getenv("DB_PORT")            
BANCO = os.getenv("DB_NAME")  

string_conexao = f"postgresql://{USUARIO}:{SENHA}@{HOST}:{PORTA}/{BANCO}"
engine = create_engine(string_conexao)

In [ ]:
df_final['TotalCharges'] = pd.to_numeric(df_final['TotalCharges'], errors='coerce')
df_final['TotalCharges'] = df_final['TotalCharges'].fillna(df_final['TotalCharges'].median())

# dim_clientes
colunas_clientes = ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents']

df_clientes = df_final[colunas_clientes].rename(columns={
    'customerID': 'customer_id',
    'SeniorCitizen': 'senior_citizen',
    'Partner': 'partner',
    'Dependents': 'dependents'
})

# Remove duplicadas por segurança
df_clientes = df_clientes.drop_duplicates(subset=['customer_id'])

print("Enviando dados para dim_clientes...")
df_clientes.to_sql('dim_clientes', con=engine, if_exists='append', index=False)

# dim_servicos
colunas_servicos = [
    'customerID', 'PhoneService', 'MultipleLines', 'InternetService', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
    'StreamingTV', 'StreamingMovies'
]
df_servicos = df_final[colunas_servicos].rename(columns={
    'customerID': 'customer_id',
    'PhoneService': 'phone_service',
    'MultipleLines': 'multiple_lines',
    'InternetService': 'internet_service',
    'OnlineSecurity': 'online_security',
    'OnlineBackup': 'online_backup',
    'DeviceProtection': 'device_protection',
    'TechSupport': 'tech_support',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_movies'
})
df_servicos = df_servicos.drop_duplicates(subset=['customer_id'])

print("Enviando dados para dim_servicos...")
df_servicos.to_sql('dim_servicos', con=engine, if_exists='append', index=False)

# fato_contratos
colunas_fato = [
    'customerID', 'tenure', 'Contract', 'PaperlessBilling', 
    'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Probability'
]
df_fato = df_final[colunas_fato].rename(columns={
    'customerID': 'customer_id',
    'Contract': 'contract',
    'PaperlessBilling': 'paperless_billing',
    'PaymentMethod': 'payment_method',
    'MonthlyCharges': 'monthly_charges',
    'TotalCharges': 'total_charges',
    'Churn': 'churn',
    'Churn_Probability': 'churn_probability'
})

print("Enviando dados para fato_contratos...")
df_fato.to_sql('fato_contratos', con=engine, if_exists='append', index=False)

print("Banco de dados preenchido com sucesso.")

Enviando dados para dim_clientes...
Enviando dados para dim_servicos...
Enviando dados para fato_contratos...
Banco de dados preenchido com sucesso em formato Star Schema!
